# 01 · Base de conhecimento, busca e geração fundamentada

**AgroFácil GenAI: assistente de segurança climática para o pequeno produtor rural.**

Este notebook cobre os itens 01 a 04 da rubrica:

1. Carrega a **base de conhecimento** (`base/trechos.jsonl`: trechos com id, fonte, URL e licença) e os **casos de teste com gabarito** (`avaliacao/casos_teste.json`, incluindo perguntas "não consta").
2. Mede a **busca** com **recall@1 e recall@3**, comparando o encoder original (inglês) com um encoder multilíngue.
3. Escolhe e **justifica o limiar** de recusa usando um conjunto de **validação** (`base/treino_lora.jsonl`), e não o conjunto de teste, para não "decorar" o teste.
4. Demonstra a **geração fundamentada**: o LLM responde em **JSON válido**, **cita o id do trecho** e **recusa** quando a informação não consta.

> Execute no Google Colab com GPU T4 (Ambiente de execução → Alterar tipo → T4 GPU).

In [ ]:
# Configuração do ambiente
# No Colab, o projeto fica no Google Drive para que todos os notebooks (01, 02, 04, 03)
# usem os mesmos arquivos: o limiar salvo no 01 e o adaptador LoRA treinado no 02.
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/palomari/agrofacil-genai.git"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive")
except ImportError:
    pass  # execução local

if not Path("src/agrofacil.py").exists():
    if Path("../src/agrofacil.py").exists():
        os.chdir("..")
    else:
        if not Path("agrofacil-genai").exists():
            subprocess.run(["git", "clone", REPO], check=True)
        os.chdir("agrofacil-genai")

assert Path("src/agrofacil.py").exists(), (
    "Versão antiga do repositório nesta pasta. Apague a pasta agrofacil-genai do Drive e rode de novo.")
sys.path.insert(0, os.getcwd())
print("Raiz do projeto:", os.getcwd())
print("Versão:", subprocess.run(["git", "log", "-1", "--format=%h %cd"], capture_output=True, text=True).stdout.strip())

In [ ]:
%pip install -q -r requirements.txt

## 1. Base de conhecimento e casos de teste

In [ ]:
import json
import numpy as np
import pandas as pd
from src import agrofacil as af

trechos = af.carregar_base()
testes = af.carregar_testes()
validacao = af.carregar_jsonl(af.CAMINHO_TREINO)

print(f"Trechos na base: {len(trechos)}")
print(f"Casos de teste: {len(testes)} "
      f"({sum(c['tipo']=='esperado' for c in testes)} com resposta na base, "
      f"{sum(c['tipo']=='nao_consta' for c in testes)} 'não consta')")
print(f"Perguntas de validação/treino: {len(validacao)}")

# Garantia de que nenhuma pergunta de treino/validação aparece no teste
sobrepostas = {af.normalizar(v["pergunta"]) for v in validacao} & {af.normalizar(c["pergunta"]) for c in testes}
assert not sobrepostas, sobrepostas
print("Sem sobreposição entre treino/validação e teste.")

In [ ]:
# Tabela da base: fonte, formato, licença e número de trechos (usada no README)
df_base = (pd.DataFrame(trechos)
           .groupby(["fonte", "url", "licenca"], as_index=False)
           .agg(trechos=("id", "count"), ids=("id", lambda s: ", ".join(s))))
df_base.insert(1, "formato", "Texto parafraseado em JSONL")
display(df_base[["fonte", "formato", "licenca", "trechos", "ids"]])
print("Total de trechos:", df_base.trechos.sum())

## 2. Busca vetorial: recall@1 e recall@3

Para cada pergunta com resposta na base, verificamos se o trecho correto (`doc_ids` do gabarito) aparece em 1º lugar (**recall@1**) ou entre os 3 primeiros (**recall@3**). Comparamos:

- `all-MiniLM-L6-v2`: encoder usado na versão anterior, treinado só em inglês;
- `paraphrase-multilingual-MiniLM-L12-v2`: encoder multilíngue, adequado a perguntas em português.

In [ ]:
ENCODERS = {
    "all-MiniLM-L6-v2 (versão anterior, inglês)": "sentence-transformers/all-MiniLM-L6-v2",
    "paraphrase-multilingual-MiniLM-L12-v2 (escolhido)": af.ENCODER_PADRAO,
}

def medir_recall(buscador, casos, k=3):
    linhas = []
    for c in casos:
        if c["tipo"] != "esperado":
            continue
        rec = buscador.buscar(c["pergunta"], k=k)
        ids = [t["id"] for t in rec]
        linhas.append({
            "id": c["id"], "pergunta": c["pergunta"], "esperado": "/".join(c["doc_ids"]),
            "top3": ids, "score_top1": round(rec[0]["score"], 3),
            "hit@1": ids[0] in c["doc_ids"], "hit@3": bool(set(ids) & set(c["doc_ids"])),
        })
    return pd.DataFrame(linhas)

buscadores, tabela, detalhes = {}, [], {}
for nome, modelo in ENCODERS.items():
    b = af.Buscador(trechos, modelo)
    buscadores[nome] = b
    det = medir_recall(b, testes)
    detalhes[nome] = det
    tabela.append({"encoder": nome, "recall@1": round(det["hit@1"].mean(), 3),
                   "recall@3": round(det["hit@3"].mean(), 3), "perguntas": len(det)})

df_recall = pd.DataFrame(tabela)
df_recall.to_csv("avaliacao/recall.csv", index=False)
df_recall

In [ ]:
# Onde a busca errou (encoder escolhido)
escolhido = "paraphrase-multilingual-MiniLM-L12-v2 (escolhido)"
det = detalhes[escolhido]
falhas = det[~det["hit@1"]]
print(f"Falhas no top-1: {len(falhas)} de {len(det)}")
display(falhas[["id", "pergunta", "esperado", "top3", "score_top1", "hit@3"]])

antigo, novo = df_recall.iloc[0], df_recall.iloc[1]
print(f"Interpretação: o encoder multilíngue tem recall@1 = {novo['recall@1']:.2f} contra {antigo['recall@1']:.2f} "
      f"do encoder em inglês, e recall@3 = {novo['recall@3']:.2f} contra {antigo['recall@3']:.2f}. "
      f"O recall@3 é o que importa para o LLM, porque ele recebe os 3 trechos no contexto.")

## 3. Limiar de recusa (escolhido na validação, verificado no teste)

O limiar é a **primeira camada de recusa**: se o trecho mais parecido tem similaridade menor que o limiar, o sistema responde "Não consta na base de conhecimento." sem chamar o LLM. A segunda camada é o próprio LLM, instruído (e, no notebook 02, treinado) a recusar quando os trechos não trazem a resposta.

**Critério:** como o LLM ainda pode recusar depois, o limiar **não deve bloquear perguntas válidas**. Escolhemos, no conjunto de validação, o **maior limiar que mantém pelo menos 95% das perguntas respondíveis**. Assim maximizamos as recusas de perguntas fora da base sem cortar perguntas legítimas.

In [ ]:
b = buscadores[escolhido]

def top1(casos):
    return [(c["tipo"], b.buscar(c["pergunta"], k=1)[0]["score"]) for c in casos]

val = [{"pergunta": v["pergunta"], "tipo": "esperado" if v["fonte_id"] else "nao_consta"} for v in validacao]
s_val, s_test = top1(val), top1(testes)

linhas = []
for lim in np.round(np.arange(0.10, 0.851, 0.025), 3):
    cob = np.mean([s >= lim for t, s in s_val if t == "esperado"])
    rec = np.mean([s < lim for t, s in s_val if t == "nao_consta"])
    linhas.append({"limiar": lim, "cobertura_respondiveis": round(cob, 3),
                   "recusa_fora_da_base": round(rec, 3), "media": round((cob + rec) / 2, 3)})
df_lim = pd.DataFrame(linhas)
df_lim.to_csv("avaliacao/limiar.csv", index=False)

candidatos = df_lim[df_lim.cobertura_respondiveis >= 0.95]
LIMIAR = float(candidatos.limiar.max()) if len(candidatos) else float(df_lim.sort_values("media").limiar.iloc[-1])
print(f"Limiar escolhido na validação: {LIMIAR}")
display(df_lim[(df_lim.limiar >= LIMIAR - 0.1) & (df_lim.limiar <= LIMIAR + 0.1)])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3.5))
for tipo, cor, rotulo in [("esperado", "#2e8b57", "com resposta na base"), ("nao_consta", "#c0392b", "não consta")]:
    ax.hist([s for t, s in s_val + s_test if t == tipo], bins=20, alpha=.6, color=cor, label=rotulo)
ax.axvline(LIMIAR, color="black", ls="--", label=f"limiar = {LIMIAR}")
ax.set_xlabel("similaridade do trecho mais parecido (cosseno)")
ax.set_ylabel("perguntas")
ax.legend()
ax.set_title("Distribuição das similaridades (validação + teste)")
plt.tight_layout()
plt.savefig("avaliacao/limiar.png", dpi=120)
plt.show()

cob_t = np.mean([s >= LIMIAR for t, s in s_test if t == "esperado"])
rec_t = np.mean([s < LIMIAR for t, s in s_test if t == "nao_consta"])
print(f"No TESTE, com limiar {LIMIAR}: {cob_t:.0%} das perguntas respondíveis passam e "
      f"{rec_t:.0%} das perguntas 'não consta' já são recusadas pelo limiar.")
print("As perguntas fora da base que passam pelo limiar (por exemplo, preço de adubo, que é do tema agrícola) "
      "dependem da segunda camada: o LLM precisa recusar. Isso é medido no notebook 04.")

with open(af.CAMINHO_CONFIG, "w", encoding="utf-8") as f:
    json.dump({"encoder": af.ENCODER_PADRAO, "limiar": LIMIAR, "k": 3}, f, indent=2)
print("Configuração salva em", af.CAMINHO_CONFIG)

## 4. Geração fundamentada: JSON válido, citação da fonte e recusa

O LLM local (`Qwen2.5-1.5B-Instruct`, quantizado em 4 bits) recebe os 3 trechos recuperados e deve responder **somente** com:

```json
{"resposta": "...", "fonte_id": "T06", "status": "ok"}
```

ou, se os trechos não trazem a resposta:

```json
{"resposta": "Não consta na base de conhecimento.", "fonte_id": null, "status": "nao_consta"}
```

O sistema **valida** o JSON e confere se o `fonte_id` citado está entre os trechos recuperados (`citacao_valida`). A fonte e a URL exibidas ao cidadão vêm da base, e não do texto gerado.

In [ ]:
model, tok = af.carregar_modelo(adapter=None)  # modelo base; o LoRA entra no notebook 02/04
cfg = af.carregar_config()

exemplos = [
    "A que horas devo ligar o pivô central para proteger a lavoura da geada?",   # tem na base
    "Como faço para receber os alertas da Defesa Civil no meu celular?",          # tem na base
    "Qual o preço atual do saco de adubo nitrogenado no mercado de São Paulo?",   # não consta (passa o limiar)
    "Quantos deputados federais o estado de São Paulo possui?",                   # não consta (barrado no limiar)
]
for p in exemplos:
    r = af.responder(p, model, tok, buscador=b, modo="rag", limiar=cfg["limiar"])
    print("Pergunta:", p)
    print("Saída do sistema:", json.dumps(af.saida_publica(r), ensure_ascii=False, indent=2))
    print(f"JSON válido: {r['json_valido']} | citação válida: {r['citacao_valida']} | origem: {r['origem']}")
    print("Texto bruto do modelo:", r["bruto"])
    print("-" * 100)

## Conclusão deste notebook

- A base tem trechos curtos e de tema único, cada um com **fonte, URL e licença**, sobre alertas, geada, irrigação na seca, chuva excessiva e temporais.
- O **recall@1/@3** foi medido por id do trecho (não por palavra-chave) e comparado entre dois encoders (`avaliacao/recall.csv`).
- O **limiar** foi escolhido em dados de validação com um critério explícito e verificado no teste (`avaliacao/limiar.csv`, `avaliacao/limiar.png`, `avaliacao/config_rag.json`).
- A geração devolve **JSON validado**, **citação conferida** contra os trechos recuperados e **recusa** em duas camadas.

O placar completo das três configurações está no notebook `04_avaliacao.ipynb`.